# Three-SMU accepted-only analysis

By default this notebook loads only completed, accepted, clean formal samples. It uses plain filesystem paths so it works over SSH or a mounted network share; rejected runs and problem samples require separate explicit audit switches.

In [ ]:
from pathlib import Path
from IPython.display import display
import matplotlib.pyplot as plt

from attodry_control.three_smu_analysis import (
    build_map, discover_three_smu_runs, load_three_smu_rows, plot_bias_iv,
    plot_gate_leakage, plot_gate_transfer, plot_map, plot_time_trace,
)


In [ ]:
DATA_DIRECTORY = Path('../data/three_smu')  # local path or SSH-mounted path
RUN_PATH = None  # optional explicit run directory, metadata.json, or data.csv
INCLUDE_REJECTED = False
INCLUDE_PROBLEM = False
SEGMENT = None
ROLE = None

available_runs = discover_three_smu_runs(DATA_DIRECTORY, include_rejected=INCLUDE_REJECTED)
for item in available_runs:
    print(f'{item.started_at}  {item.status:11}  {item.run_name}  {item.run_dir}')
selected = Path(RUN_PATH) if RUN_PATH else (available_runs[0].run_dir if available_runs else None)
if selected is None:
    raise RuntimeError('No completed accepted run found; set RUN_PATH or check DATA_DIRECTORY.')
rows = load_three_smu_rows(
    selected, include_rejected=INCLUDE_REJECTED,
    include_problem=INCLUDE_PROBLEM, segment=SEGMENT, role=ROLE,
)
print(f'Loaded {len(rows)} long-form rows.')


In [ ]:
# Choose the plot matching the scan mode. These functions never open hardware.
figure = plot_bias_iv(rows)
display(figure)
plt.close(figure)


In [ ]:
# Example alternatives:
# figure = plot_gate_transfer(rows, gate_role='gate_top')
# figure = plot_time_trace(rows, role='smu_bias', field='current_a')
# figure = plot_gate_leakage(rows)
# A two-gate map with a fixed bias: 
# grid = build_map(rows, x_role='gate_top', y_role='gate_bottom',
#                  fixed_coordinates={'smu_bias': 0.001})
# figure = plot_map(grid)
